# LeetCode 1073 — Adding Two Negabinary Numbers bằng Python

Link đề: https://leetcode.com/problems/adding-two-negabinary-numbers/ · Độ khó: **Medium** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def addNegabinary(self, arr1: List[int], arr2: List[int]) -> List[int]:
        ...
```

Ghi chú: đề yêu cầu trả về mảng **không có số `0` đứng đầu** (hoặc đúng `[0]`). Với điều kiện đó, biểu diễn cơ số `-2` của một số nguyên là **duy nhất**, nên đáp án là **duy nhất** và ta so sánh trực tiếp mảng kết quả. Điểm cốt lõi: cộng từng chữ số từ bit thấp nhất như cộng nhị phân, nhưng **số nhớ có thể là `-1`, `0`, `1`** vì `2·(-2)^i = -(-2)^{i+1}`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu hệ cơ số `-2` (negabinary): chữ số `0/1`, trọng số `(-2)^i`; mảng cho từ bit **cao** đến bit **thấp**.
2. Nắm công cụ: duyệt từ bit thấp, số nhớ có dấu, chia sàn (`>>` / `//`) với số âm, bỏ số `0` đứng đầu, chuyển đổi nguyên ↔ cơ số `-2`.
3. Đi từ **chuyển sang số nguyên, cộng, chuyển ngược** → **cộng từng chữ số có số nhớ** → **bảng chuyển trạng thái** → **mẹo mặt nạ bit** → **cộng thô rồi chuẩn hoá**.
4. Hiểu mấu chốt: `a + b + carry ∈ {-1, 0, 1, 2, 3}`; bit kết quả `= tổng & 1`, số nhớ mới `= -(tổng >> 1)` (vì một đơn vị "2" ở bit `i` bằng `-1` ở bit `i+1`).
5. Tránh lỗi kinh điển: dùng số nhớ nhị phân thường (`+1`), chia cắt về `0` khi tổng `= -1`, quên số nhớ còn lại ở cuối, quên bỏ số `0` đầu, trả `[]` cho kết quả `0`, căn lề mảng sai.

## 2. Đề bài & ví dụ

Cho hai số `arr1` và `arr2` ở cơ số **-2**. Mỗi số là một mảng `0`/`1` từ bit **quan trọng nhất** đến bit **ít quan trọng nhất**. Ví dụ `arr = [1,1,0,1]` biểu diễn `(-2)^3 + (-2)^2 + (-2)^0 = -8 + 4 + 1 = -3`. Mảng **không có số `0` đứng đầu**: hoặc `arr == [0]` hoặc `arr[0] == 1`.

Trả về tổng của `arr1` và `arr2` ở **cùng định dạng** (mảng `0`/`1`, không có số `0` đứng đầu).

**Ví dụ 1:** `arr1 = [1,1,1,1,1]`, `arr2 = [1,0,1]` → `[1,0,0,0,0]` (`11 + 5 = 16`).

**Ví dụ 2:** `arr1 = [0]`, `arr2 = [0]` → `[0]`.

**Ví dụ 3:** `arr1 = [0]`, `arr2 = [1]` → `[1]`.

**Ràng buộc:** `1 <= arr1.length, arr2.length <= 1000`; phần tử `0` hoặc `1`; không có số `0` đứng đầu.

In [1]:
import itertools
import random
import time
from typing import List

examples = [
    ([1, 1, 1, 1, 1], [1, 0, 1], [1, 0, 0, 0, 0]),
    ([0], [0], [0]),
    ([0], [1], [1]),
]
for a, b, want in examples:
    print(f"arr1 = {a}, arr2 = {b} -> {want}")

arr1 = [1, 1, 1, 1, 1], arr2 = [1, 0, 1] -> [1, 0, 0, 0, 0]
arr1 = [0], arr2 = [0] -> [0]
arr1 = [0], arr2 = [1] -> [1]


## 3. Kiến thức nền

### 3.1 Giá trị của mảng cơ số `-2`

Với `arr = [d_{L-1}, …, d_1, d_0]` (từ cao đến thấp), giá trị `= Σ d_i · (-2)^i`. Tính bằng Horner: `v = 0; for d in arr: v = v·(-2) + d`.

In [2]:
def neg_to_int(arr):
    v = 0
    for d in arr:
        v = v * (-2) + d
    return v


print(neg_to_int([1, 1, 0, 1]), neg_to_int([1, 1, 1, 1, 1]), neg_to_int([1, 0, 1]), neg_to_int([1, 0, 0, 0, 0]))
assert neg_to_int([1, 1, 0, 1]) == -3 and neg_to_int([1, 1, 1, 1, 1]) == 11 and neg_to_int([1, 0, 1]) == 5

-3 11 5 16


### 3.2 Đổi số nguyên sang cơ số `-2`

Lấy chữ số thấp nhất `r = n mod 2 ∈ {0, 1}`; phần còn lại `n ← (n - r) / (-2)`. Lặp đến khi `n = 0`. Mỗi số nguyên có **đúng một** biểu diễn không có số `0` đứng đầu (kết quả `0` là `[0]`).

In [3]:
def int_to_neg(n):
    if n == 0:
        return [0]
    digits = []
    while n != 0:
        r = n & 1                                  # n mod 2 trong {0, 1}, đúng cả với n âm trong Python
        digits.append(r)
        n = (n - r) // -2                          # (n - r) chẵn nên chia -2 chính xác
    return digits[::-1]


for v in range(-12, 13):
    assert neg_to_int(int_to_neg(v)) == v
print({v: int_to_neg(v) for v in (-3, -2, -1, 0, 1, 2, 3, 11, 16)})

{-3: [1, 1, 0, 1], -2: [1, 0], -1: [1, 1], 0: [0], 1: [1], 2: [1, 1, 0], 3: [1, 1, 1], 11: [1, 1, 1, 1, 1], 16: [1, 0, 0, 0, 0]}


### 3.3 Tính duy nhất

Các mảng độ dài `≤ L` không có số `0` đứng đầu đại diện cho đúng `2^L` số nguyên **đôi một khác nhau** (nằm liên tiếp trong khoảng). Vì vậy đáp án của bài là duy nhất và ta kiểm tra bằng so sánh mảng trực tiếp.

In [4]:
def all_arrays(max_len):
    yield [0]
    for L in range(1, max_len + 1):
        for tail in itertools.product((0, 1), repeat=L - 1):
            yield [1, *tail]


vals = [neg_to_int(a) for a in all_arrays(8)]
print(len(vals), "mảng độ dài <= 8:", "giá trị từ", min(vals), "đến", max(vals))
assert len(set(vals)) == len(vals) == 256 and sorted(vals) == list(range(min(vals), max(vals) + 1))

256 mảng độ dài <= 8: giá trị từ -170 đến 85


## 4. Ý tưởng nền tảng / trực giác

Cộng như ở trường: từ bit thấp lên, mỗi bước cộng hai chữ số và số nhớ. Trong nhị phân, tổng `2` ở bit `i` thành `0` nhớ `+1` sang bit `i+1` vì `2 · 2^i = 2^{i+1}`. Trong cơ số `-2`, `2 · (-2)^i = -(-2)^{i+1}`, nên tổng `2` ở bit `i` thành `0` và nhớ **`-1`** sang bit `i+1`. Số nhớ âm khi cộng vào có thể cho tổng `-1`: không thể là chữ số, ta viết `-1 = 1 + (-2)` tức chữ số `1` nhớ `+1`. Do đó số nhớ chỉ thuộc `{-1, 0, 1}`.

## 5. Brute force — đổi sang số nguyên, cộng, đổi ngược

**Ý tưởng:** tính `x = neg_to_int(arr1)`, `y = neg_to_int(arr2)`, cộng số nguyên, rồi `int_to_neg(x + y)`. Đúng theo định nghĩa (và nhờ tính duy nhất ở mục 3.3 cho đúng đáp án). Python có số nguyên lớn nên chạy được với `n = 1000` (số `~2^1000`). Dùng làm **oracle** vì không đụng vào quy tắc số nhớ.

In [5]:
def add_via_int(arr1, arr2):
    return int_to_neg(neg_to_int(arr1) + neg_to_int(arr2))


for a, b, want in examples:
    assert add_via_int(a, b) == want
print([add_via_int(a, b) for a, b, _ in examples])

[[1, 0, 0, 0, 0], [0], [1]]


In [6]:
# Code nộp LeetCode cho Brute force (đổi sang số nguyên lớn; đúng, nhưng dựa vào số nguyên lớn của Python)
class SolutionViaInt:
    def addNegabinary(self, arr1: List[int], arr2: List[int]) -> List[int]:
        def to_int(arr):
            v = 0
            for d in arr:
                v = v * (-2) + d
            return v

        n = to_int(arr1) + to_int(arr2)
        if n == 0:
            return [0]
        digits = []
        while n != 0:
            r = n & 1
            digits.append(r)
            n = (n - r) // -2
        return digits[::-1]

**Độ phức tạp:** `O(n²)` thời gian (phép nhân/chia số nguyên lớn `n` bit lặp `n` lần), `O(n)` bộ nhớ. Ngôn ngữ không có số nguyên lớn (Java, C++) không dùng được cách này.

## 6. Bước đột phá

Không cần đổi sang số nguyên: cộng **từng chữ số** từ bit thấp với **số nhớ có dấu** `carry ∈ {-1, 0, 1}`. Tổng `s = a_i + b_i + carry ∈ [-1, 3]`; chữ số kết quả `s & 1`, số nhớ mới `-(s >> 1)`. Cuối cùng bỏ số `0` đứng đầu. `O(n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** cộng từng chữ số có số nhớ (đáp án chính), **Cách 3** bảng chuyển trạng thái, **Cách 4** mẹo mặt nạ bit trên số nguyên, **Cách 5** cộng thô rồi chuẩn hoá.

### Cách 2 — Cộng từng chữ số có số nhớ (đáp án chính)

**Giải thích:** duyệt `i` từ bit thấp (cuối mảng) lên; `s = a_i + b_i + carry`; chữ số `s & 1`; `carry = -(s >> 1)`. Tiếp tục tới khi hết cả hai mảng **và** `carry = 0`. Đảo kết quả rồi bỏ số `0` đứng đầu (giữ lại ít nhất `[0]`).

In [7]:
def add_carry(arr1, arr2):
    i, j = len(arr1) - 1, len(arr2) - 1
    carry = 0
    out = []                                          # từ bit thấp đến cao
    while i >= 0 or j >= 0 or carry:
        s = carry
        if i >= 0:
            s += arr1[i]
            i -= 1
        if j >= 0:
            s += arr2[j]
            j -= 1
        out.append(s & 1)                             # chữ số: s mod 2 (cũng đúng khi s = -1: -1 & 1 = 1)
        carry = -(s >> 1)                             # s >> 1 là chia sàn: s = -1 -> -1 -> carry = 1; s = 2 -> 1 -> carry = -1
    while len(out) > 1 and out[-1] == 0:
        out.pop()                                     # bỏ số 0 đứng đầu, giữ lại [0]
    return out[::-1] or [0]


for a, b, want in examples:
    assert add_carry(a, b) == want
print([add_carry(a, b) for a, b, _ in examples])

[[1, 0, 0, 0, 0], [0], [1]]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionCarry:
    def addNegabinary(self, arr1: List[int], arr2: List[int]) -> List[int]:
        i, j = len(arr1) - 1, len(arr2) - 1
        carry = 0
        out = []
        while i >= 0 or j >= 0 or carry:
            s = carry
            if i >= 0:
                s += arr1[i]
                i -= 1
            if j >= 0:
                s += arr2[j]
                j -= 1
            out.append(s & 1)
            carry = -(s >> 1)
        while len(out) > 1 and out[-1] == 0:
            out.pop()
        return out[::-1] or [0]

**Độ phức tạp:** `O(max(n1, n2))` thời gian (kết quả dài tối đa `max + 2`), `O(max(n1, n2))` bộ nhớ cho kết quả.

### Cách 3 — Bảng chuyển trạng thái

**Ý tưởng:** tổng `s = a_i + b_i + carry` chỉ có `5` giá trị `-1..3`; lập sẵn bảng `s → (chữ số, số nhớ mới)`:

| `s` | chữ số | số nhớ mới | kiểm tra: `chữ số + (-2)·nhớ` |
|---|---|---|---|
| `-1` | `1` | `+1` | `1 + (-2)(+1) = -1` |
| `0` | `0` | `0` | `0` |
| `1` | `1` | `0` | `1` |
| `2` | `0` | `-1` | `0 + (-2)(-1) = 2` |
| `3` | `1` | `-1` | `1 + (-2)(-1) = 3` |

Mỗi hàng thoả `s = chữ số + (-2) · số nhớ mới` — chính là điều kiện để giá trị bảo toàn.

In [9]:
TABLE = {-1: (1, 1), 0: (0, 0), 1: (1, 0), 2: (0, -1), 3: (1, -1)}
for s, (digit, nc) in TABLE.items():
    assert digit + (-2) * nc == s and digit in (0, 1) and nc in (-1, 0, 1)      # bảo toàn giá trị


def add_table(arr1, arr2):
    x, y = arr1[::-1], arr2[::-1]
    out, carry = [], 0
    for k in range(max(len(x), len(y)) + 2):           # +2: kết quả dài tối đa max + 2
        s = carry + (x[k] if k < len(x) else 0) + (y[k] if k < len(y) else 0)
        digit, carry = TABLE[s]
        out.append(digit)
    while len(out) > 1 and out[-1] == 0:
        out.pop()
    assert carry == 0
    return out[::-1]


for a, b, want in examples:
    assert add_table(a, b) == want
print([add_table(a, b) for a, b, _ in examples])

[[1, 0, 0, 0, 0], [0], [1]]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionTable:
    def addNegabinary(self, arr1: List[int], arr2: List[int]) -> List[int]:
        table = {-1: (1, 1), 0: (0, 0), 1: (1, 0), 2: (0, -1), 3: (1, -1)}
        x, y = arr1[::-1], arr2[::-1]
        out, carry = [], 0
        for k in range(max(len(x), len(y)) + 2):
            s = carry + (x[k] if k < len(x) else 0) + (y[k] if k < len(y) else 0)
            digit, carry = table[s]
            out.append(digit)
        while len(out) > 1 and out[-1] == 0:
            out.pop()
        return out[::-1]

**Độ phức tạp:** `O(max(n1, n2))` thời gian và bộ nhớ.

### Cách 4 — Mẹo mặt nạ bit trên số nguyên

**Ý tưởng:** nếu `M` là số có các bit ở vị trí **lẻ** bật (`…101010₂`) thì với `x` là mảng bit đọc như số nhị phân, **giá trị cơ số `-2`** là `(x XOR M) − M`; ngược lại số nguyên `v` có mảng cơ số `-2` là `(v + M) XOR M`. Vậy cộng = đổi sang số nguyên bằng phép XOR/trừ, cộng, đổi ngược — toàn phép toán trên số nguyên lớn, không vòng lặp theo chữ số (nhưng vẫn `O(n)` bit mỗi phép).

In [11]:
def odd_mask(width):
    return sum(1 << i for i in range(1, width, 2))


def add_mask(arr1, arr2):
    width = max(len(arr1), len(arr2)) + 4
    M = odd_mask(width)
    x = int("".join(map(str, arr1)), 2)
    y = int("".join(map(str, arr2)), 2)
    total = ((x ^ M) - M) + ((y ^ M) - M)
    z = (total + M) ^ M
    return [int(c) for c in bin(z)[2:]]                 # bin(0) = '0b0' -> [0]


for a, b, want in examples:
    assert add_mask(a, b) == want
for a in list(all_arrays(6)):                           # kiểm công thức chuyển đổi
    M = odd_mask(10)
    assert ((int("".join(map(str, a)), 2) ^ M) - M) == neg_to_int(a)
print([add_mask(a, b) for a, b, _ in examples])

[[1, 0, 0, 0, 0], [0], [1]]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionMask:
    def addNegabinary(self, arr1: List[int], arr2: List[int]) -> List[int]:
        width = max(len(arr1), len(arr2)) + 4
        M = sum(1 << i for i in range(1, width, 2))
        x = int("".join(map(str, arr1)), 2)
        y = int("".join(map(str, arr2)), 2)
        total = ((x ^ M) - M) + ((y ^ M) - M)
        z = (total + M) ^ M
        return [int(c) for c in bin(z)[2:]]

**Độ phức tạp:** `O(n)` phép toán trên số nguyên `n` bit (`O(n)` mỗi phép ⇒ `O(n)` tổng cộng với hằng số nhỏ), `O(n)` bộ nhớ. Hiệu quả trong Python nhờ số nguyên lớn.

### Cách 5 — Cộng thô rồi chuẩn hoá

**Ý tưởng:** bước 1 cộng **không nhớ** từng cột: `d_i = a_i + b_i ∈ {0, 1, 2}`. Bước 2 **chuẩn hoá** từ thấp lên cao: với `d_i` (đã cộng số nhớ từ bước trước) lấy `r = d_i mod 2`, số nhớ `c = -(d_i - r)/2` cộng vào `d_{i+1}`. Hai bước tách bạch, dễ giải thích; cùng kết quả với Cách 2.

In [13]:
def add_normalize(arr1, arr2):
    L = max(len(arr1), len(arr2)) + 3
    d = [0] * L
    for k, v in enumerate(arr1[::-1]):
        d[k] += v
    for k, v in enumerate(arr2[::-1]):
        d[k] += v                                        # bước 1: cộng thô từng cột, 0..2
    for k in range(L - 1):                               # bước 2: chuẩn hoá từ thấp lên cao
        r = d[k] % 2
        d[k + 1] += -((d[k] - r) // 2)
        d[k] = r
    out = d[::-1]
    while len(out) > 1 and out[0] == 0:
        out.pop(0)
    return out


for a, b, want in examples:
    assert add_normalize(a, b) == want
print([add_normalize(a, b) for a, b, _ in examples])

[[1, 0, 0, 0, 0], [0], [1]]


In [14]:
# Code nộp LeetCode cho Cách 5
class SolutionNormalize:
    def addNegabinary(self, arr1: List[int], arr2: List[int]) -> List[int]:
        L = max(len(arr1), len(arr2)) + 3
        d = [0] * L
        for k, v in enumerate(arr1[::-1]):
            d[k] += v
        for k, v in enumerate(arr2[::-1]):
            d[k] += v
        for k in range(L - 1):
            r = d[k] % 2
            d[k + 1] += -((d[k] - r) // 2)
            d[k] = r
        out = d[::-1]
        while len(out) > 1 and out[0] == 0:
            out.pop(0)
        return out

**Độ phức tạp:** `O(n)` thời gian (bước `pop(0)` đầu mảng có thể `O(n²)` trong trường hợp xấu nhưng kết quả chỉ dư tối đa `3` số `0`), `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace **Cách 2** với `arr1 = [1,1,1,1,1]` (`11`), `arr2 = [1,0,1]` (`5`). Duyệt từ bit thấp:

| bước `i` | `a_i` | `b_i` | `carry` vào | `s = a + b + carry` | chữ số `s & 1` | `carry` ra `= -(s >> 1)` |
|---|---|---|---|---|---|---|
| 0 | 1 | 1 | 0 | 2 | 0 | -1 |
| 1 | 1 | 0 | -1 | 0 | 0 | 0 |
| 2 | 1 | 1 | 0 | 2 | 0 | -1 |
| 3 | 1 | 0 | -1 | 0 | 0 | 0 |
| 4 | 1 | 0 | 0 | 1 | 1 | 0 |

Chữ số từ thấp đến cao `0,0,0,0,1` ⇒ đảo `[1,0,0,0,0]` = `(-2)^4 = 16 = 11 + 5`. Hàm trace in từng bước:

In [15]:
def trace_carry(arr1, arr2):
    i, j = len(arr1) - 1, len(arr2) - 1
    carry, out, step = 0, [], 0
    print(f"arr1 = {arr1} ({neg_to_int(arr1)}), arr2 = {arr2} ({neg_to_int(arr2)})")
    while i >= 0 or j >= 0 or carry:
        a = arr1[i] if i >= 0 else 0
        b = arr2[j] if j >= 0 else 0
        i, j = i - 1, j - 1
        s = a + b + carry
        digit, new_carry = s & 1, -(s >> 1)
        print(f"  bước {step}: a = {a}, b = {b}, carry = {carry:+d} -> s = {s:+d}, chữ số {digit}, carry mới = {new_carry:+d}")
        out.append(digit)
        carry = new_carry
        step += 1
    while len(out) > 1 and out[-1] == 0:
        out.pop()
    res = out[::-1]
    print("  kết quả:", res, "=", neg_to_int(res))
    return res


assert trace_carry([1, 1, 1, 1, 1], [1, 0, 1]) == [1, 0, 0, 0, 0]
assert trace_carry([1], [1]) == [1, 1, 0]            # 1 + 1 = 2 = (-2)^2 + (-2)^1 = 4 - 2

arr1 = [1, 1, 1, 1, 1] (11), arr2 = [1, 0, 1] (5)
  bước 0: a = 1, b = 1, carry = +0 -> s = +2, chữ số 0, carry mới = -1
  bước 1: a = 1, b = 0, carry = -1 -> s = +0, chữ số 0, carry mới = +0
  bước 2: a = 1, b = 1, carry = +0 -> s = +2, chữ số 0, carry mới = -1
  bước 3: a = 1, b = 0, carry = -1 -> s = +0, chữ số 0, carry mới = +0
  bước 4: a = 1, b = 0, carry = +0 -> s = +1, chữ số 1, carry mới = +0
  kết quả: [1, 0, 0, 0, 0] = 16
arr1 = [1] (1), arr2 = [1] (1)
  bước 0: a = 1, b = 1, carry = +0 -> s = +2, chữ số 0, carry mới = -1
  bước 1: a = 0, b = 0, carry = -1 -> s = -1, chữ số 1, carry mới = +1
  bước 2: a = 0, b = 0, carry = +1 -> s = +1, chữ số 1, carry mới = +0
  kết quả: [1, 1, 0] = 2


**Trace `1 + 1`:** `s = 2` ⇒ chữ số `0`, `carry = -1`; bước 1: `s = -1` ⇒ chữ số `1`, `carry = +1`; bước 2: `s = 1` ⇒ chữ số `1`, `carry = 0`. Mảng `[1,1,0]` = `4 - 2 + 0 = 2`. Ở đây xuất hiện số nhớ `-1` rồi `+1`: lý do cần ba giá trị cho `carry`.

## 9. Code nộp LeetCode

Chọn **Cách 2** (cộng từng chữ số có số nhớ). Giữ nguyên `class Solution`, `addNegabinary(self, arr1, arr2)`:

In [16]:
class Solution:
    def addNegabinary(self, arr1: List[int], arr2: List[int]) -> List[int]:
        i, j = len(arr1) - 1, len(arr2) - 1         # bắt đầu từ bit thấp nhất (cuối mảng)
        carry = 0                                   # số nhớ thuộc {-1, 0, 1}
        out = []                                    # kết quả từ bit thấp đến cao
        while i >= 0 or j >= 0 or carry:
            s = carry
            if i >= 0:
                s += arr1[i]
                i -= 1
            if j >= 0:
                s += arr2[j]
                j -= 1
            out.append(s & 1)                       # chữ số hiện tại
            carry = -(s >> 1)                       # 2·(-2)^k = -(-2)^{k+1}: số nhớ đổi dấu
        while len(out) > 1 and out[-1] == 0:        # bỏ số 0 đứng đầu (cuối danh sách), giữ lại [0]
            out.pop()
        return out[::-1]


print([Solution().addNegabinary(a, b) for a, b, _ in examples])
assert all(Solution().addNegabinary(a, b) == v for a, b, v in examples)

[[1, 0, 0, 0, 0], [0], [1]]


## 10. Vì sao đúng

**(a) Bất biến giá trị.** Gọi `A_k, B_k` là giá trị của `k` chữ số thấp nhất của hai số, `R_k` là giá trị của `k` chữ số kết quả đã sinh. Ta chứng minh bằng quy nạp: `A_k + B_k = R_k + c_k · (-2)^k`, trong đó `c_k` là số nhớ sau `k` bước. Với `k = 0`: `0 = 0 + 0`. Bước `k → k+1`: `s = a_k + b_k + c_k`, chữ số `d = s & 1`, `c_{k+1} = -(s - d)/2` (vì `s >> 1 = (s - d)/2` khi `d = s & 1`). Khi đó `d + (-2) · c_{k+1} = d + (s - d) = s`. Do đó `A_{k+1} + B_{k+1} = A_k + B_k + (a_k + b_k)(-2)^k = R_k + (c_k + a_k + b_k)(-2)^k = R_k + s·(-2)^k = R_k + d·(-2)^k + (-2)·c_{k+1}·(-2)^k = R_{k+1} + c_{k+1}·(-2)^{k+1}`. ∎

**(b) Số nhớ thuộc `{-1, 0, 1}`.** Nếu `c ∈ {-1,0,1}` và `a, b ∈ {0,1}` thì `s ∈ [-1, 3]`; bảng ở Cách 3 cho `-(s >> 1) ∈ {1, 0, 0, -1, -1}` (với `s = -1, 0, 1, 2, 3`) nên số nhớ mới cũng thuộc `{-1, 0, 1}`.

**(c) Kết thúc.** Sau khi hết cả hai mảng, số nhớ còn lại tiếp tục sinh chữ số: `c = 1`: `s = 1` ⇒ chữ số `1`, nhớ `0`; `c = -1`: `s = -1` ⇒ chữ số `1`, nhớ `+1` ⇒ bước sau `s = 1` ⇒ chữ số `1`, nhớ `0`. Vậy tối đa thêm `2` chữ số: kết quả dài `≤ max(n1, n2) + 2`, vòng lặp luôn dừng.

**(d) Đúng giá trị và duy nhất.** Khi hết số và `c = 0`, theo (a) `A + B = R`. Chuỗi chữ số có thể có số `0` đứng đầu; bỏ chúng không đổi giá trị. Theo mục 3.3 biểu diễn không có số `0` đứng đầu của một số nguyên là duy nhất, nên đây chính là đáp án mong đợi (và kết quả `0` là `[0]`).

**(e) Các cách còn lại.** Cách 1 là định nghĩa + tính duy nhất; Cách 3 chỉ tra bảng của (b); Cách 4 dựa vào công thức `(x XOR M) − M` (mỗi bit lẻ có trọng số `-2^i = 2^i − 2·2^i` nên XOR với `M` rồi trừ `M` cho đúng `Σ d_i (-2)^i`); Cách 5 tách việc cộng và chuẩn hoá nhưng cùng quy tắc `d_k = r + (-2)·c`.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp oracle (đổi sang số nguyên) trên **mọi cặp** số có `≤ 8` chữ số (`256² = 65 536` cặp) và trên các cặp ngẫu nhiên dài tới `1000` chữ số (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đổi sang số nguyên | `O(n²)` | `O(n)` | oracle; cần số nguyên lớn |
| 2. Cộng có số nhớ | `O(n)` | `O(n)` | **đáp án chính** |
| 3. Bảng chuyển | `O(n)` | `O(n)` | dễ kiểm chứng |
| 4. Mặt nạ bit | `O(n)` (số nguyên lớn) | `O(n)` | gọn; chỉ hợp ngôn ngữ có số nguyên lớn |
| 5. Cộng thô + chuẩn hoá | `O(n)` | `O(n)` | hai bước dễ giải thích |

Đo thực tế (`n = 1000`):

In [17]:
def rand_neg(rng, length):
    return [1] + [rng.randint(0, 1) for _ in range(length - 1)]


def bench(fn, a, b, repeat=5):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(a, b)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1073)
a, b = rand_neg(rng, 1000), rand_neg(rng, 1000)
for name, fn in [("via_int", add_via_int), ("carry", add_carry), ("table", add_table), ("mask", add_mask), ("normalize", add_normalize)]:
    r = fn(a, b)
    print(f"{name:10s}: {bench(fn, a, b) * 1000:7.2f} ms -> độ dài {len(r)}")

via_int   :    0.36 ms -> độ dài 1000
carry     :    0.18 ms -> độ dài 1000
table     :    0.15 ms -> độ dài 1000
mask      :    0.32 ms -> độ dài 1000
normalize :    0.27 ms -> độ dài 1000


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi cặp nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "carry": add_carry,
    "table": add_table,
    "mask": add_mask,
    "normalize": add_normalize,
    "Solution": lambda a, b: Solution().addNegabinary(a[:], b[:]),
    "SolutionCarry": lambda a, b: SolutionCarry().addNegabinary(a[:], b[:]),
    "SolutionTable": lambda a, b: SolutionTable().addNegabinary(a[:], b[:]),
    "SolutionMask": lambda a, b: SolutionMask().addNegabinary(a[:], b[:]),
    "SolutionNormalize": lambda a, b: SolutionNormalize().addNegabinary(a[:], b[:]),
}
ORACLES = {"via_int": add_via_int, "SolutionViaInt": lambda a, b: SolutionViaInt().addNegabinary(a[:], b[:])}


def is_normalized(arr):
    return arr == [0] or (len(arr) >= 1 and arr[0] == 1 and all(v in (0, 1) for v in arr))


def check(a, b, want=None):
    if want is None:
        want = add_via_int(a, b)
    assert is_normalized(want)
    for name, fn in {**IMPLS, **ORACLES}.items():
        got = fn(a, b)
        assert got == want, (name, a, b, got, want)
        assert is_normalized(got) and neg_to_int(got) == neg_to_int(a) + neg_to_int(b)
    return want


for a, b, want in examples:
    check(a, b, want)
print("ví dụ đề: mọi cài đặt đúng và đầu ra chuẩn (không có số 0 đứng đầu);", len(IMPLS) + len(ORACLES), "cài đặt")

ví dụ đề: mọi cài đặt đúng và đầu ra chuẩn (không có số 0 đứng đầu); 11 cài đặt


### B. Edge case

- `[0] + [0]` → `[0]`; `[0] + a = a`.
- `[1] + [1] = [1,1,0]` (số nhớ `-1` rồi `+1`).
- Tổng bằng `0` từ hai số khác `0` (ví dụ `[1,1]` là `-1` và `[1]` là `1`): kết quả `[0]`.
- Các độ dài khác nhau; số dài `1000` cộng số ngắn.
- Số nhớ kéo dài (`[1,1,1,1,1,1,1,1]` + `[1]`).
- Tổng làm kết quả dài hơn `max + 1` chữ số (cần tới `+2`).
- Số âm lớn cộng số âm lớn.

In [19]:
edge = [
    ([0], [0], [0]), ([0], [1], [1]), ([1], [0], [1]), ([1], [1], [1, 1, 0]), ([1, 1], [1], [0]),
    ([1, 1, 0, 1], [1, 1, 0, 1], None), ([1] * 8, [1], None), ([1, 0, 1], [1, 1, 1], None),
    ([1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], None),
    ([1] + [0] * 999, [1] + [0] * 999, None), ([1] * 1000, [1], None), ([1] * 1000, [1] * 1000, None),
]
for a, b, want in edge:
    got = check(a, b, want)
    print(f"  n1 = {len(a):4d}, n2 = {len(b):4d}: kết quả độ dài {len(got)}", got if len(got) <= 8 else "")
assert add_carry([1], [1]) == [1, 1, 0] and neg_to_int([1, 1, 0]) == 2
assert add_carry([1, 1], [1]) == [0] and neg_to_int([1, 1]) == -1
print(len(edge), "edge case: mọi cài đặt khớp")

  n1 =    1, n2 =    1: kết quả độ dài 1 [0]
  n1 =    1, n2 =    1: kết quả độ dài 1 [1]
  n1 =    1, n2 =    1: kết quả độ dài 1 [1]
  n1 =    1, n2 =    1: kết quả độ dài 3 [1, 1, 0]
  n1 =    2, n2 =    1: kết quả độ dài 1 [0]
  n1 =    4, n2 =    4: kết quả độ dài 4 [1, 1, 1, 0]
  n1 =    8, n2 =    1: kết quả độ dài 8 [1, 1, 1, 1, 1, 1, 0, 0]
  n1 =    3, n2 =    3: kết quả độ dài 5 [1, 1, 0, 0, 0]
  n1 =    8, n2 =    8: kết quả độ dài 8 [1, 0, 1, 0, 1, 0, 1, 0]
  n1 = 1000, n2 = 1000: kết quả độ dài 1002 
  n1 = 1000, n2 =    1: kết quả độ dài 1000 
  n1 = 1000, n2 = 1000: kết quả độ dài 1000 
12 edge case: mọi cài đặt khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_binary_carry(arr1, arr2):                       # số nhớ nhị phân thường: carry = s // 2
    i, j, carry, out = len(arr1) - 1, len(arr2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry + (arr1[i] if i >= 0 else 0) + (arr2[j] if j >= 0 else 0)
        i, j = i - 1, j - 1
        out.append(s & 1)
        carry = s >> 1
    while len(out) > 1 and out[-1] == 0:
        out.pop()
    return out[::-1]


def bug_truncating_division(arr1, arr2):                # chia cắt về 0 (kiểu C): int(s / 2)
    i, j, carry, out = len(arr1) - 1, len(arr2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry + (arr1[i] if i >= 0 else 0) + (arr2[j] if j >= 0 else 0)
        i, j = i - 1, j - 1
        out.append(s & 1)
        carry = -int(s / 2)
    while len(out) > 1 and out[-1] == 0:
        out.pop()
    return out[::-1]


def bug_drop_final_carry(arr1, arr2):                   # dừng khi hết chữ số, bỏ số nhớ còn lại
    i, j, carry, out = len(arr1) - 1, len(arr2) - 1, 0, []
    while i >= 0 or j >= 0:
        s = carry + (arr1[i] if i >= 0 else 0) + (arr2[j] if j >= 0 else 0)
        i, j = i - 1, j - 1
        out.append(s & 1)
        carry = -(s >> 1)
    while len(out) > 1 and out[-1] == 0:
        out.pop()
    return out[::-1]


def bug_leading_zeros(arr1, arr2):                      # không bỏ số 0 đứng đầu
    i, j, carry, out = len(arr1) - 1, len(arr2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry + (arr1[i] if i >= 0 else 0) + (arr2[j] if j >= 0 else 0)
        i, j = i - 1, j - 1
        out.append(s & 1)
        carry = -(s >> 1)
    return out[::-1]


def bug_empty_zero(arr1, arr2):                         # bỏ hết số 0 đầu, kể cả khi kết quả là 0
    i, j, carry, out = len(arr1) - 1, len(arr2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry + (arr1[i] if i >= 0 else 0) + (arr2[j] if j >= 0 else 0)
        i, j = i - 1, j - 1
        out.append(s & 1)
        carry = -(s >> 1)
    while out and out[-1] == 0:
        out.pop()
    return out[::-1]


def bug_left_align(arr1, arr2):                         # căn lề trái thay vì căn theo bit thấp
    n = max(len(arr1), len(arr2))
    x = arr1 + [0] * (n - len(arr1))
    y = arr2 + [0] * (n - len(arr2))
    return add_carry(x, y)


BUGS = [bug_binary_carry, bug_truncating_division, bug_drop_final_carry, bug_leading_zeros, bug_empty_zero, bug_left_align]
pool = list(all_arrays(5))
for bug in BUGS:
    found = next(((a, b) for a in pool for b in pool if bug(a, b) != add_via_int(a, b)), None)
    assert found, bug.__name__
    a, b = found
    print(f"{bug.__name__:24s} sai trên {a} + {b}: ra {bug(a, b)}, đúng {add_via_int(a, b)}")

bug_binary_carry         sai trên [1] + [1]: ra [1, 0], đúng [1, 1, 0]
bug_truncating_division  sai trên [1] + [1]: ra [1, 0], đúng [1, 1, 0]
bug_drop_final_carry     sai trên [1] + [1]: ra [0], đúng [1, 1, 0]
bug_leading_zeros        sai trên [1] + [1, 1]: ra [0, 0], đúng [0]
bug_empty_zero           sai trên [0] + [0]: ra [], đúng [0]
bug_left_align           sai trên [1] + [1, 0]: ra [1, 1, 0, 0], đúng [1, 1]


### D. Vét cạn mọi cặp nhỏ

Mọi cặp `(arr1, arr2)` với độ dài `≤ 8` (không có số `0` đứng đầu): `256 × 256 = 65 536` cặp. So mọi cài đặt với oracle (đổi sang số nguyên) và kiểm đầu ra chuẩn.

In [21]:
t0 = time.perf_counter()
arrs = list(all_arrays(8))
cnt = 0
for a in arrs:
    va = neg_to_int(a)
    for b in arrs:
        want = int_to_neg(va + neg_to_int(b))
        for name, fn in IMPLS.items():
            if name.startswith("Solution"):
                continue
            assert fn(a, b) == want, (name, a, b)
        cnt += 1
print(cnt, "cặp (độ dài <= 8):", len(IMPLS) - 5, "hàm demo khớp oracle;", round(time.perf_counter() - t0, 1), "s")
t0 = time.perf_counter()
sub = random.Random(1).sample(arrs, 60)
for a in sub:
    for b in sub:
        check(a, b)
print("60 × 60 cặp: cả các lớp Solution* khớp oracle;", round(time.perf_counter() - t0, 1), "s")

65536 cặp (độ dài <= 8): 4 hàm demo khớp oracle; 1.2 s


60 × 60 cặp: cả các lớp Solution* khớp oracle; 0.3 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Cặp ngẫu nhiên độ dài tới `1000`: mọi cách khớp oracle; (2) **giao hoán** `a + b = b + a`; (3) **đơn vị** `a + [0] = a`; (4) **kết hợp** `(a + b) + c = a + (b + c)`; (5) **độ dài** `≤ max + 2`; (6) **phần tử đối**: `a + neg(a) = [0]` với `neg(a) = add(a·(-2), a)` (xem mục 15); (7) giá trị bằng tổng giá trị.

In [22]:
def shift(arr):                                          # nhân với -2: thêm một số 0 cuối mảng
    return arr + [0] if arr != [0] else [0]


def negate(arr):                                         # -a = (-2)a + a
    return add_carry(shift(arr), arr)


rng = random.Random(20240173)
t0 = time.perf_counter()
for it in range(400):
    n1, n2, n3 = rng.randint(1, 1000), rng.randint(1, 1000), rng.randint(1, 1000)
    a = [0] if rng.random() < 0.05 else rand_neg(rng, n1)
    b = [0] if rng.random() < 0.05 else rand_neg(rng, n2)
    c = rand_neg(rng, n3)
    s = check(a, b)                                                                   # (1), (7)
    assert Solution().addNegabinary(b, a) == s                                        # (2)
    assert Solution().addNegabinary(a, [0]) == a                                      # (3)
    assert Solution().addNegabinary(s, c) == Solution().addNegabinary(a, Solution().addNegabinary(b, c))   # (4)
    assert len(s) <= max(len(a), len(b)) + 2                                          # (5)
    assert Solution().addNegabinary(a, negate(a)) == [0] and neg_to_int(negate(a)) == -neg_to_int(a)       # (6)
print("400 cặp độ dài tới 1000: 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

400 cặp độ dài tới 1000: 7 tính chất đúng; 1.9 s


### F. Giới hạn

`n = 1000`: số `1` lặp (`[1]*1000`), `[1]` rồi toàn `0` (luỹ thừa của `-2`), cặp toàn `1`, số dài cộng số ngắn, cặp đối nhau (tổng `[0]`).

In [23]:
t0 = time.perf_counter()
rg = random.Random(9)
big = [
    ([1] * 1000, [1] * 1000), ([1] + [0] * 999, [1] + [0] * 999), ([1] * 1000, [1]), ([1] + [0] * 999, [1] * 999),
    (rand_neg(rg, 1000), rand_neg(rg, 1000)), (rand_neg(rg, 1000), [0]),
]
a = rand_neg(rg, 1000)
big.append((a, negate(a)))
for a, b in big:
    r = check(a, b)
    print(f"  n1 = {len(a)}, n2 = {len(b)}: kết quả độ dài {len(r)}")
assert Solution().addNegabinary(*big[-1]) == [0]
print("n = 1000:", round(time.perf_counter() - t0, 1), "s")

  n1 = 1000, n2 = 1000: kết quả độ dài 1000
  n1 = 1000, n2 = 1000: kết quả độ dài 1002
  n1 = 1000, n2 = 1: kết quả độ dài 1000
  n1 = 1000, n2 = 999: kết quả độ dài 1000
  n1 = 1000, n2 = 1000: kết quả độ dài 1000
  n1 = 1000, n2 = 1: kết quả độ dài 1000
  n1 = 1000, n2 = 1001: kết quả độ dài 1
n = 1000: 0.1 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng số nhớ nhị phân thường

In [24]:
print("bug:", bug_binary_carry([1], [1]), "| đúng:", Solution().addNegabinary([1], [1]))
assert bug_binary_carry([1], [1]) == [1, 0] and Solution().addNegabinary([1], [1]) == [1, 1, 0]
assert neg_to_int([1, 0]) == -2 and neg_to_int([1, 1, 0]) == 2

bug: [1, 0] | đúng: [1, 1, 0]


**Vì sao sai:** `[1,0]` có giá trị `-2`, không phải `2`. Trong cơ số `-2`, `1 + 1 = 2 = 4 - 2 = [1,1,0]`; số nhớ phải đổi dấu: `2·(-2)^i = -(-2)^{i+1}`.

### Lỗi 2: chia cắt về `0` khi tổng `= -1`

In [25]:
bad = next((a, b) for a in pool for b in pool if bug_truncating_division(a, b) != add_via_int(a, b))
print("phản ví dụ:", bad, "| bug:", bug_truncating_division(*bad), "| đúng:", add_via_int(*bad))
assert bug_truncating_division(*bad) != add_via_int(*bad)

phản ví dụ: ([1], [1]) | bug: [1, 0] | đúng: [1, 1, 0]


**Vì sao sai:** khi `s = -1`: chữ số `1` và số nhớ phải là `+1` (vì `-1 = 1 + (-2)·1`). Chia cắt `int(-1 / 2) = 0` cho số nhớ `0`, sai. Dùng **chia sàn** (`s >> 1` hoặc `s // 2`) trong Python; ở C/C++/Java cần xử lý riêng tổng âm.

### Lỗi 3: bỏ số nhớ còn lại ở cuối

In [26]:
print("bug:", bug_drop_final_carry([1], [1]), "| đúng:", Solution().addNegabinary([1], [1]))
assert bug_drop_final_carry([1], [1]) != Solution().addNegabinary([1], [1])

bug: [0] | đúng: [1, 1, 0]


**Vì sao sai:** vòng lặp phải chạy **đến khi hết cả hai mảng và `carry = 0`**; kết quả có thể dài hơn `max(n1, n2)` tới `2` chữ số.

### Lỗi 4: không bỏ số `0` đứng đầu

In [27]:
print("bug:", bug_leading_zeros([1, 1], [1]), "| đúng:", Solution().addNegabinary([1, 1], [1]))
assert bug_leading_zeros([1, 1], [1]) != [0] and Solution().addNegabinary([1, 1], [1]) == [0]

bug: [0, 0] | đúng: [0]


**Vì sao sai:** đề yêu cầu mảng không có số `0` đứng đầu (trừ `[0]`). Khi hai số triệt tiêu, vòng lặp sinh các số `0` thừa như `[0, 0, 0]` cần cắt về `[0]`.

### Lỗi 5: trả về `[]` khi kết quả bằng `0`

In [28]:
print("bug:", bug_empty_zero([0], [0]), "| đúng:", Solution().addNegabinary([0], [0]))
assert bug_empty_zero([0], [0]) == [] and Solution().addNegabinary([0], [0]) == [0]

bug: [] | đúng: [0]


**Vì sao sai:** biểu diễn của `0` là `[0]`, không phải mảng rỗng. Khi cắt số `0` đầu phải giữ lại ít nhất một phần tử (`while len(out) > 1 ...`).

### Lỗi 6 (bonus): căn lề trái thay vì căn theo bit thấp

In [29]:
bad = next((a, b) for a in pool for b in pool if bug_left_align(a, b) != add_via_int(a, b))
print("phản ví dụ:", bad, "| bug:", bug_left_align(*bad), "| đúng:", add_via_int(*bad))
assert bug_left_align(*bad) != add_via_int(*bad)

phản ví dụ: ([1], [1, 0]) | bug: [1, 1, 0, 0] | đúng: [1, 1]


**Vì sao sai:** mảng cho từ bit **cao** đến bit **thấp**; cộng phải căn theo bit **thấp** (cuối mảng). Đệm số `0` ở cuối mảng làm đổi giá trị (nhân với `(-2)^k`).

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "cộng hai số ở cơ số lạ (âm, `-2`, `-k`, cân bằng ba phân...) trên mảng chữ số" → cộng từng chữ số có số nhớ, **xác định số nhớ từ đẳng thức bảo toàn giá trị** `s = chữ số + cơ số · số nhớ mới`.

**Mẹo:**

1. Duyệt từ bit thấp nhất (cuối mảng) lên.
2. `chữ số = s & 1`, `carry = -(s >> 1)`; dựa vào chia sàn của Python.
3. Dừng khi `i < 0` và `j < 0` và `carry == 0`.
4. Bỏ số `0` đứng đầu nhưng giữ lại `[0]`.
5. Oracle: đổi sang số nguyên lớn của Python, cộng, đổi ngược; kiểm tính duy nhất.
6. Bảng chuyển `s → (chữ số, nhớ)` giúp chứng minh và debug.

**Họ bài cùng pattern:** Convert to Base -2 (1017), Add Binary (67), Add Two Numbers (2), Add Strings (415), Plus One (66), Multiply Strings (43).

## 15. Biến thể

1. **Số đối** `-a = (-2)·a + a`: nhân `-2` là thêm một số `0` ở cuối mảng.
2. **Phép trừ** `a - b = a + (-b)`.
3. **Phép nhân** `a · b = Σ b_i · (-2)^i · a` (dịch và cộng).
4. **Đổi số nguyên sang cơ số `-2`** (bài 1017) — đã làm ở mục 3.2.

Cài đặt và đối chiếu số nguyên:

In [30]:
def sub_neg(a, b):
    return add_carry(a, negate(b))


def mul_neg(a, b):
    result = [0]
    for k, bit in enumerate(b[::-1]):                    # b = Σ bit_k (-2)^k
        if bit:
            term = a + [0] * k if a != [0] else [0]      # a · (-2)^k: thêm k số 0
            result = add_carry(result, term)
    return result


rng = random.Random(15)
for _ in range(1500):
    a = rng.choice(arrs)
    b = rng.choice(arrs)
    assert neg_to_int(sub_neg(a, b)) == neg_to_int(a) - neg_to_int(b) and is_normalized(sub_neg(a, b))
    assert neg_to_int(mul_neg(a, b)) == neg_to_int(a) * neg_to_int(b) and is_normalized(mul_neg(a, b))
    assert sub_neg(a, b) == int_to_neg(neg_to_int(a) - neg_to_int(b))
for v in range(-300, 301):
    assert neg_to_int(int_to_neg(v)) == v and is_normalized(int_to_neg(v))
print("số đối / phép trừ / phép nhân / đổi số nguyên: khớp số nguyên")

số đối / phép trừ / phép nhân / đổi số nguyên: khớp số nguyên


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đổi số nguyên | chuyển, cộng, chuyển ngược | `O(n²)` | `O(n)` | oracle (cần số nguyên lớn) |
| 2. Cộng có số nhớ | `s & 1`, `-(s >> 1)` | `O(n)` | `O(n)` | **đáp án chính** |
| 3. Bảng chuyển | tra bảng `5` trạng thái | `O(n)` | `O(n)` | dễ chứng minh |
| 4. Mặt nạ bit | `(x ^ M) - M` | `O(n)` | `O(n)` | Python, rất ngắn |
| 5. Cộng thô + chuẩn hoá | hai bước | `O(n)` | `O(n)` | dễ giải thích |

Mọi cách khớp oracle trên `65 536` cặp số nhỏ (`≤ 8` chữ số) và `400` cặp ngẫu nhiên dài tới `1000` chữ số.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().addNegabinary([1, 1], [1, 1]), Solution().addNegabinary([1, 0], [1, 0]), Solution().addNegabinary([1, 1, 0, 1], [1]), Solution().addNegabinary([1], [1, 1]))

[1, 0] [1, 1, 0, 0] [1, 0] [0]


### Bài 2 (tính tay)

`arr1 = [1,0,1]` (`5`) và `arr2 = [1,1,1]` (`3`). Lập bảng từng bước (`a`, `b`, `carry`, `s`, chữ số, `carry` mới) và cho biết kết quả (kiểm bằng giá trị). Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cộng có số nhớ:

In [32]:
def add_todo(arr1, arr2):
    i, j, carry, out = len(arr1) - 1, len(arr2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry
        pass            # TODO 1: cộng arr1[i], arr2[j] (nếu còn) vào s và lùi i, j
        pass            # TODO 2: out.append(s & 1); carry = -(s >> 1)
    return out[::-1]    # TODO 3: bỏ số 0 đứng đầu (giữ lại [0])


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(add_todo(a, b) == w for a, b, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **bảng chuyển trạng thái**:

In [33]:
class MySolution:
    def addNegabinary(self, arr1, arr2):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().addNegabinary(a, b) == w for a, b, w in examples)

### Bài 5 (mini challenge): chuyển số nguyên sang cơ số `-2`

Viết `to_negabinary(n)` trả về mảng cơ số `-2` (không số `0` đứng đầu) của số nguyên `n` (bài 1017). Gợi ý: lặp `r = n & 1`, `n = (n - r) // -2`. Kiểm tra: `to_negabinary(2) == [1,1,0]`, `to_negabinary(3) == [1,1,1]`, `to_negabinary(0) == [0]`, `to_negabinary(-3) == [1,1,0,1]`.

In [34]:
def to_negabinary(n):
    pass            # TODO


# Khi làm xong:
# assert to_negabinary(2) == [1, 1, 0] and to_negabinary(3) == [1, 1, 1] and to_negabinary(0) == [0] and to_negabinary(-3) == [1, 1, 0, 1]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `[1,1] = -1`, `[1,0] = -2`, `[1,1,0,1] = -3`.
- Mức 2: `-1 + -1 = -2 = [1,0]`; `-2 + -2 = -4 = [1,1,0,0]`; `-3 + 1 = -2 = [1,0]`; `1 + -1 = 0 = [0]`.
- Mức 3: `[1,0] [1,1,0,0] [1,0] [0]`.

**Bài 2**

- Mức 1: từ bit thấp: `1 + 1`, `0 + 1`, `1 + 1`.
- Mức 2: bước 0: `s = 2`, chữ số `0`, `carry = -1`; bước 1: `s = 0 + 1 - 1 = 0`, chữ số `0`, `carry = 0`; bước 2: `s = 2`, chữ số `0`, `carry = -1`; bước 3: `s = -1`, chữ số `1`, `carry = +1`; bước 4: `s = 1`, chữ số `1`, `carry = 0`.
- Mức 3: chữ số từ thấp đến cao `0,0,0,1,1` ⇒ `[1,1,0,0,0]` `= -8 + 16 = 8 = 5 + 3`.

**Bài 3**

- Mức 1: `if i >= 0: s += arr1[i]; i -= 1` và tương tự `j`.
- Mức 2: `out.append(s & 1)`; `carry = -(s >> 1)`.
- Mức 3: `while len(out) > 1 and out[-1] == 0: out.pop()` trước khi đảo.

**Bài 4**

- Mức 1: bảng `{-1: (1, 1), 0: (0, 0), 1: (1, 0), 2: (0, -1), 3: (1, -1)}`.
- Mức 2: lặp `max(len) + 2` lần trên mảng đảo, cộng `0` khi hết chữ số.
- Mức 3: bỏ số `0` đứng đầu, đảo lại.

**Bài 5**

- Mức 1: `n == 0` trả `[0]`.
- Mức 2: `r = n & 1`; thêm `r`; `n = (n - r) // -2`.
- Mức 3: đảo danh sách chữ số ở cuối.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
vals = (Solution().addNegabinary([1, 1], [1, 1]), Solution().addNegabinary([1, 0], [1, 0]), Solution().addNegabinary([1, 1, 0, 1], [1]), Solution().addNegabinary([1], [1, 1]))
print(vals)
assert vals == ([1, 0], [1, 1, 0, 0], [1, 0], [0])
assert all(v == add_via_int(a, b) for v, (a, b) in zip(vals, [([1, 1], [1, 1]), ([1, 0], [1, 0]), ([1, 1, 0, 1], [1]), ([1], [1, 1])]))
print("Bài 1 OK")

([1, 0], [1, 1, 0, 0], [1, 0], [0])
Bài 1 OK


### Bài 2

In [36]:
r = trace_carry([1, 0, 1], [1, 1, 1])
assert r == [1, 1, 0, 0, 0] and neg_to_int(r) == 8 == 5 + 3
print("Bài 2 OK")

arr1 = [1, 0, 1] (5), arr2 = [1, 1, 1] (3)
  bước 0: a = 1, b = 1, carry = +0 -> s = +2, chữ số 0, carry mới = -1
  bước 1: a = 0, b = 1, carry = -1 -> s = +0, chữ số 0, carry mới = +0
  bước 2: a = 1, b = 1, carry = +0 -> s = +2, chữ số 0, carry mới = -1
  bước 3: a = 0, b = 0, carry = -1 -> s = -1, chữ số 1, carry mới = +1
  bước 4: a = 0, b = 0, carry = +1 -> s = +1, chữ số 1, carry mới = +0
  kết quả: [1, 1, 0, 0, 0] = 8
Bài 2 OK


### Bài 3

In [37]:
def add_answer(arr1, arr2):
    i, j, carry, out = len(arr1) - 1, len(arr2) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry
        if i >= 0:
            s += arr1[i]
            i -= 1
        if j >= 0:
            s += arr2[j]
            j -= 1
        out.append(s & 1)
        carry = -(s >> 1)
    while len(out) > 1 and out[-1] == 0:
        out.pop()
    return out[::-1]


assert all(add_answer(a, b) == w for a, b, w in examples)
for a in pool:
    for b in pool:
        assert add_answer(a, b) == add_via_int(a, b)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def addNegabinary(self, arr1, arr2):
        table = {-1: (1, 1), 0: (0, 0), 1: (1, 0), 2: (0, -1), 3: (1, -1)}
        x, y = arr1[::-1], arr2[::-1]
        out, carry = [], 0
        for k in range(max(len(x), len(y)) + 2):
            s = carry + (x[k] if k < len(x) else 0) + (y[k] if k < len(y) else 0)
            d, carry = table[s]
            out.append(d)
        while len(out) > 1 and out[-1] == 0:
            out.pop()
        return out[::-1]


assert all(MySolutionAnswer().addNegabinary(a, b) == w for a, b, w in examples)
for a in pool:
    for b in pool:
        assert MySolutionAnswer().addNegabinary(a, b) == add_via_int(a, b)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def to_negabinary(n):
    if n == 0:
        return [0]
    digits = []
    while n != 0:
        r = n & 1
        digits.append(r)
        n = (n - r) // -2
    return digits[::-1]


assert to_negabinary(2) == [1, 1, 0] and to_negabinary(3) == [1, 1, 1] and to_negabinary(0) == [0] and to_negabinary(-3) == [1, 1, 0, 1]
for v in range(-1000, 1001):
    assert neg_to_int(to_negabinary(v)) == v and is_normalized(to_negabinary(v))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (cộng hai số mảng chữ số ở cơ số `-2`):**

```python
class Solution:
    def addNegabinary(self, arr1: List[int], arr2: List[int]) -> List[int]:
        i, j, carry, out = len(arr1) - 1, len(arr2) - 1, 0, []
        while i >= 0 or j >= 0 or carry:
            s = carry
            if i >= 0:
                s += arr1[i]
                i -= 1
            if j >= 0:
                s += arr2[j]
                j -= 1
            out.append(s & 1)
            carry = -(s >> 1)
        while len(out) > 1 and out[-1] == 0:
            out.pop()
        return out[::-1]
```

**Checklist trước khi submit:**

- [ ] Duyệt từ **bit thấp** (cuối mảng) và đảo kết quả ở cuối?
- [ ] `chữ số = s & 1`, `carry = -(s >> 1)` (chia **sàn**, số nhớ có thể `-1`, `0`, `1`)?
- [ ] Vòng lặp chạy khi còn `i`, `j` **hoặc** `carry`?
- [ ] Bỏ số `0` đứng đầu nhưng giữ lại `[0]`?
- [ ] Đã thử `[1]+[1] → [1,1,0]`, `[1,1]+[1] → [0]`, `[0]+[0] → [0]`?
- [ ] Đúng tên `class Solution`, `addNegabinary(self, arr1, arr2)`, không in debug.